# 09 — Robustness: how fragile is notebook 08's picture?

Three checks, registered in §1. Does a longer estimation window (504 or 756 days instead of 252) change
the Sharpe ratios of GMV, MDP, ERC and HRP (H3)? Do the eight core runs keep their Sharpe ranking from the
train to the test window (H5, descriptive)? Is the 252-day volatility forecast that every method relies on
calibrated (H4)? The last check sets up the volatility overlay of notebook 10. The hypotheses are numbered
H3–H5 because notebook 08 used H1–H2.

## Setup

Data as in notebook 08: same ETFs, rebalances, costs and cash. Run labels carry the covariance estimator, (S) for sample; BL(0.1,S) is notebook 03's BL(0.1). The §4 grid runs add the lookback in days, e.g. GMV(S) 504d.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
import _nb07_helpers as h7
import _nb08_helpers as h8
import _nb09_helpers as h
from maplab import Panel, apply_style

In [ ]:
apply_style()

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"

## §1 What we test

Registered before any statistic below was computed; the frozen text is in
`registrations/nb09.toml`. Newey–West lags and the bootstrap seed are
notebook 08's.

In plain words: H3 asks whether estimating the covariance matrix on two or three years of data instead of one changes a method's Sharpe ratio. H5 asks whether the runs that ranked high before the split still rank high after it. H4 asks whether the risk forecast every method relies on is too high, that is, whether realized variance comes in below the forecast on average (v̄, the mean ratio of realized to forecast variance, below 1).

| | Question | Test | Verdict rule |
|---|---|---|---|
| H3 (8 tests) | Do GMV, MDP, ERC and HRP have a different Sharpe ratio at 504 or 756 days than at 252 days, all scored from 2011-01-31? | Sharpe-difference test with Newey–West errors; Holm adjustment over the 8 | Rejected if Holm p < 0.05 and the test-window difference has the same sign |
| H5 (descriptive) | Do the eight core runs keep their Sharpe ranking from train to test? | Spearman rank correlation | – |
| H4 (8 tests) | Does the 252-day ex-ante volatility over-forecast realized variance, i.e. is the mean variance ratio v̄ below 1? | t = (v̄ − 1) / bootstrap sd, segment block bootstrap (blocks of 12 segments, 10,000 draws); no multiplicity adjustment | Supported if t ≤ −2 and the test-window v̄ is below 1; t ≤ −2 alone is not robust |

An H3 adjusted p between 0.05 and 0.10 holds "narrowly". H4 uses the variance ratio rather than the mean log ratio q of notebook 07
because q is negative even when the forecast is right (the log of an average is
above the average of logs), while v̄ is unbiased. The grid's 252-day runs start
two years after notebook 08's (2011-01-31, the first month-end with 756 days of
history), so their Sharpe ratios differ from notebook 08's by construction.

Limits: H4 measures the sample-Σ forecast at each method's weights, also for
methods whose weights use other inputs (EW, BL); notebook 07's mean q
motivated H4, but the v̄ rule was fixed before v̄ was computed; the common start
drops 2009–2010 from H3; with 8 Holm-adjusted tests, NULL HOLDS in H3 is not
evidence that lookback does not matter.

In [ ]:
reg = ml.inference.load_registration("nb09")

## §2 The statistics

General forms of what §1 adds to notebook 08. $x_t$ is a run's daily net return in excess of BIL. The Newey–West covariance, the delta-method Sharpe difference and the Holm adjustment are as defined in notebook 08 §2.

What each piece answers. The lookback ΔSR asks whether a longer window changes a method's Sharpe ratio. The variance ratio v̄ asks whether the volatility forecast is too high or too low, and the Jensen gap $J$ explains why notebook 07's mean log ratio $\bar q$ can be negative even when the forecast is right. The segment block bootstrap gives the uncertainty of v̄ without treating neighbouring segments as independent. The Spearman correlation measures whether the ranking of the runs holds from train to test.

**Lookback ΔSR on a common start.** For method $m$ and lookback $L \in \lbrace 504, 756 \rbrace$,
$$\hat\Delta_{m,L} = \widehat{SR}\big(x^{(m,L)}\big) - \widehat{SR}\big(x^{(m,252)}\big),$$
with both runs scored from 2011-01-31, so the two series cover the same days. Its standard error is the notebook 08 delta-method form with $y_t = \big(x^{(m,L)}_t,\ x^{(m,252)}_t,\ (x^{(m,L)}_t)^2,\ (x^{(m,252)}_t)^2\big)$. A method's two tests share its 252-day run, so the 8 statistics are dependent; Holm's adjustment holds under any dependence.

**Variance ratio and Jensen gap.** Holding segment $k = 1,\dots,K$ has realized vol $s_k$ and ex-ante vol $\hat\sigma_k = \sqrt{w_k'\hat\Sigma_k w_k}$ (§1). With $v_k = s_k^2/\hat\sigma_k^2$ and $q_k = \log(s_k/\hat\sigma_k) = \tfrac12\log v_k$,
$$\bar v = \frac1K\sum_{k=1}^K v_k, \qquad \bar q = \tfrac12\log\bar v - J, \qquad J = \tfrac12\Big(\log\bar v - \frac1K\sum_{k=1}^K\log v_k\Big) \ge 0 .$$
$J \ge 0$ is Jensen's inequality for the concave log, with equality only when all $v_k$ are equal. If a segment's daily returns are serially uncorrelated and their variance equals the forecast, $E[s_k^2] = \hat\sigma_k^2$, so $E[\bar v] = 1$ while $E[\bar q] < 0$: a negative $\bar q$ alone does not show an over-forecast, and $\bar v$ carries no such bias.

**Segment block bootstrap.** The $v_k$ in time order are resampled in blocks of $b = 12$ consecutive segments: $\lceil K/b\rceil$ block starts are drawn uniformly from $1,\dots,K$, each block wraps from segment $K$ back to segment 1, and the joined blocks are cut to length $K$. The blocks keep the dependence between neighbouring segments. From $B$ resampled means $\bar v^*$: $t = (\bar v - 1)/\mathrm{sd}(\bar v^*)$ and the 95% percentile interval.

**Spearman rank correlation.** With $R_i$ and $S_i$ the ranks of run $i$'s train and test Sharpe ratios, $\rho_S$ is the Pearson correlation of the ranks; without ties $\rho_S = 1 - 6\sum_i (R_i - S_i)^2/\big(n(n^2-1)\big)$. With $n = 8$ runs it is descriptive.

## §3 The core runs

The eight core runs of notebook 08 at 252 days from the default start (208 rebalances, 40 in test). They feed H5 and H4. Their Sharpe ratios match notebook 08's in all three windows; `tests/test_reproduction.py` checks this.

In [ ]:
CORE = h.CORE
core = h.build_core_runs(simple_returns, panel)
for name, r in core.items():
    assert r["wlog"].index.equals(rdates), f"{name}: wlog index != rdates"

core_summary = h8.summary_table(core, CORE, split_ts, rf_daily)

## §4 Lookback grid

GMV, MDP, ERC and HRP (sample Σ) at 252, 504 and 756 days, all scored from the common start 2011-01-31 (184 rebalances, 40 in test), with a weight cross-check against §3 and the summary table in the three windows.

In [ ]:
common_start = h.common_start(log_returns.index, 756)
cs_dates = ml.rebalance_dates(simple_returns.index)
cs_dates = cs_dates[cs_dates >= common_start]
assert common_start == pd.Timestamp("2011-01-31")
assert len(cs_dates) == 184 and int((cs_dates > split_ts).sum()) == 40

GRID = h.GRID
grid = h.build_grid_runs(simple_returns, panel, common_start)
for name, r in grid.items():
    assert r["wlog"].index.equals(cs_dates), f"{name}: wlog index != common-start rebalances"
    assert r["diag"]["n_rebalances"] == 184
print(f"common start {common_start.date()}: {len(cs_dates)} rebalances ({int((cs_dates > split_ts).sum())} in test); "
      f"{len(grid)} grid runs")

Grid 252-day target weights against the §3 target weights on the 184 common rebalance dates.

In [ ]:
dw = h.weight_crosscheck(grid, core)
assert dw.max() < 1e-12
print(f"252-day grid weights equal the §3 weights on all {len(cs_dates)} common dates (max |Δw| = {dw.max():.1e})")

**Reading.** The grid's 252-day runs differ from the core runs only in their later start: on all 184 common dates their weights are identical to the §3 weights.

Grid runs, one row each: Sharpe in the three windows of the common-start sample, the rest on its full window.

In [ ]:
grid_summary = h8.summary_table(grid, GRID, split_ts, rf_daily)

Xg = h8.excess_frame(grid, GRID, rf_daily)
XgW = h8.window_frames(Xg, split_ts)
Xg_full, Xg_test = XgW["full"], XgW["test (> split)"]
seg0 = simple_returns.index[simple_returns.index >= common_start][0]
assert Xg_full.index[0] == seg0

gf = grid_summary.xs("full", level="window")
grid_shown = pd.DataFrame({
    "Sharpe full": gf["sharpe"],
    "Sharpe train": grid_summary.xs("train (<= split)", level="window")["sharpe"],
    "Sharpe test": grid_summary.xs("test (> split)", level="window")["sharpe"],
    "return (%)": gf["ann_return"], "vol (%)": gf["ann_vol"], "max DD (%)": gf["max_dd"],
    "turnover (%/yr)": gf["ann_turnover"],
}).rename_axis(None)
ml.reporting.show_table(grid_shown, pct=["return (%)", "vol (%)", "max DD (%)", "turnover (%/yr)"],
                        pct_dp=1, int_cols=["turnover (%/yr)"])

In [ ]:
start_cmp = pd.DataFrame({
    "Sharpe, from 2011": [grid_summary.loc[(h.grid_label(m, 252), "full"), "sharpe"] for m in h.GRID_METHODS],
    "Sharpe, notebook 08 start": [core_summary.loc[(f"{m}(S)", "full"), "sharpe"] for m in h.GRID_METHODS],
}, index=h.GRID_METHODS)
start_cmp["difference"] = start_cmp["Sharpe, from 2011"] - start_cmp["Sharpe, notebook 08 start"]
ml.reporting.show_table(start_cmp)

**Reading.** The later start alone moves each method's 252-day full-window Sharpe by −0.03 (MDP) to −0.09 (ERC) against §3, the same size as the lookback differences tested in §5 (+0.06 to +0.10).

## §5 H3 — lookback

Delta-method Newey–West test of each method's 504-day and 756-day Sharpe ratio against its 252-day Sharpe ratio on the common-start full window, Holm-adjusted over the 8 tests, with the test-window difference.

In [ ]:
T_full, T_test = len(Xg_full), len(Xg_test)
rows = []
for m in h.GRID_METHODS:
    base = h.grid_label(m, 252)
    for L in (504, 756):
        lab = h.grid_label(m, L)
        r = ml.robust.sharpe_diff_hac(Xg_full[lab], Xg_full[base])
        assert r["L"] == ml.robust.nw_lag(T_full)
        rt = ml.robust.sharpe_diff_hac(Xg_test[lab], Xg_test[base])
        assert rt["L"] == ml.robust.nw_lag(T_test)
        rows.append({"id": f"H3_{m}_{L}", "method": m, "lookback": L, "dSR_full": r["d_sr"], "se": r["se"],
                     "t": r["t"], "p_raw": r["p"], "dSR_test": rt["d_sr"],
                     "same_sign_test": bool(np.sign(rt["d_sr"]) == np.sign(r["d_sr"]))})

h3 = pd.DataFrame(rows).set_index("id")
h3["p_holm"] = ml.robust.holm(h3["p_raw"].to_numpy())

h3_stats = {}
for hid, row in h3.iterrows():
    h3_stats[hid] = dict(
        display=f"ΔSR {row['dSR_full']:+.2f}, t {row['t']:.2f}, Holm p {row['p_holm']:.3f}",
        p_adj=float(row["p_holm"]), same_sign_test=bool(row["same_sign_test"]),
    )
reg_ids = {x["id"] for x in reg["hypothesis"]}
assert len(h3_stats) == 8 and set(h3_stats) <= reg_ids
assert set(h3_stats) == {x["id"] for x in reg["hypothesis"] if x["id"].startswith("H3_") and x["kind"] == "adjusted_null"}

h3_shown = h3[["dSR_full", "se", "t", "p_raw", "p_holm", "dSR_test", "same_sign_test"]].rename(columns={
    "dSR_full": "ΔSR full", "p_raw": "p", "p_holm": "Holm p", "dSR_test": "ΔSR test",
    "same_sign_test": "same sign in test"}).rename_axis(None)
h3_disp = ml.reporting.show_table(h3_shown, dp=2, col_dp={"p": 3, "Holm p": 3, "ΔSR test": 3})
h3_disp

**Reading.** No lookback difference survives the Holm adjustment. All 8 are positive (+0.06 to +0.10), and all 8 keep their sign in the test window, though ERC and HRP at 504 days gain only +0.007 and +0.003 there. The largest t is 2.03 (ERC 504d); after Holm over the 8 tests its p is 0.338, the smallest of the 8.

H3 descriptive: annual turnover, median effN (effective number of holdings, $1/\sum_i w_i^2$), median UUP target weight and UUP target weight at 2022-12-31 per method and lookback.

In [ ]:
h3_desc = h.h3_desc_table(grid, grid_summary, "2022-12-31")
h3_desc_shown = h3_desc.rename(columns={
    "ann_turnover": "turnover (%/yr)", "median_effN": "median effN", "median_w_UUP": "median UUP weight (%)",
    "w_UUP_2022-12-31": "UUP weight 2022-12-31 (%)"})
display(ml.reporting.show_table(h3_desc_shown, pct=["turnover (%/yr)", "median UUP weight (%)", "UUP weight 2022-12-31 (%)"],
                                pct_dp=1, int_cols=["turnover (%/yr)"]))

turn_cmp = pd.DataFrame({
    "turnover from 2011 (%/yr)": [grid_summary.loc[(h.grid_label(m, 252), "full"), "ann_turnover"] for m in h.GRID_METHODS],
    "turnover, notebook 08 start (%/yr)": [core_summary.loc[(f"{m}(S)", "full"), "ann_turnover"] for m in h.GRID_METHODS],
}, index=h.GRID_METHODS)
ml.reporting.show_table(turn_cmp, pct=list(turn_cmp.columns), pct_dp=1)

**Reading.** A longer lookback mainly cuts turnover. From 252 to 756 days annual turnover goes GMV 71% → 29%, MDP 50% → 24%, ERC 26% → 15%, HRP 62% → 28%, so the 756-day level is 41–57% of the 252-day level for all 4 methods. The UUP weight at 2022-12-31 falls for all 4 methods (GMV 58.1% → 46.6%, MDP 59.2% → 49.4%, ERC 48.4% → 38.5%, HRP 55.3% → 50.7%), while each method's median UUP weight over the 184 rebalances stays within 2.3 pp of its 252-day value. The common start changes 252-day turnover by at most 2.2 pp against §3 (second table).

Full-window Sharpe, annual turnover and median effective N by lookback, one line per method.

In [ ]:
sharpe_full = grid_summary.xs("full", level="window")["sharpe"]
fig = h.lookback_panels(h3_desc, sharpe_full)
plt.show()

**Reading.** Sharpe rises from 252 to 504 days for all 4 methods and falls slightly again at 756 days for GMV, MDP and ERC (HRP keeps rising, to 0.70). Turnover falls with each longer lookback for all 4 methods; median effN moves by at most 0.36 from its 252-day value (ERC at 756 days).

UUP target weight at each rebalance by lookback, one panel per method; the dotted line marks the train/test split.

In [ ]:
fig = h.uup_paths(grid, split_ts)
plt.show()

rows = []
for lab in GRID:
    wt = grid[lab]["wlog"].loc[grid[lab]["wlog"].index > split_ts]
    assert len(wt) == 40
    med = wt.median()
    rows.append({"run": lab, "median UUP weight, test (%)": float(med["UUP"]), "largest median holding": med.idxmax()})
uup_test = pd.DataFrame(rows).set_index("run").rename_axis(None)
ml.reporting.show_table(uup_test, pct=["median UUP weight, test (%)"], pct_dp=1)

**Reading.** Lookback does not change the test-window leader: over the 40 test rebalances UUP has the largest median weight in all 12 grid runs. Its test-window median weight is 45.1–54.0% across the 4 methods at 756 days and 44.4–52.8% at 252 days. Longer lookbacks smooth and delay the path: the 756-day weight reaches its 2023–24 level after the split and stays higher through 2025–26.

## §6 H5 — Sharpe ranks, train vs test (descriptive)

Train and test Sharpe ratios of the 8 §3 core runs, their ranks and the Spearman rank correlation.

In [ ]:
h5 = pd.DataFrame({
    "SR_train": core_summary.xs("train (<= split)", level="window").loc[CORE, "sharpe"],
    "SR_test": core_summary.xs("test (> split)", level="window").loc[CORE, "sharpe"],
})
h5["rank_train"] = h5["SR_train"].rank(ascending=False)
h5["rank_test"] = h5["SR_test"].rank(ascending=False)
rho_s = float(h5["SR_train"].corr(h5["SR_test"], method="spearman"))
print(f"Spearman rank correlation, train vs test: {rho_s:.2f}")
ml.reporting.show_table(h5.rename(columns={"SR_train": "Sharpe train", "SR_test": "Sharpe test",
                                           "rank_train": "rank train", "rank_test": "rank test"}).rename_axis(None),
                        int_cols=["rank train", "rank test"])

**Reading.** The ranking is only half stable: the Spearman rank correlation is 0.50, descriptive with 8 runs (no test is registered). The largest rise is MaxSharpe from 6th to 1st and the largest fall BL(0.1) from 5th to 8th; MDP keeps its rank (3rd).

Annualized volatility and return of each core run, train point (hollow) to test point (filled).

In [ ]:
fig = h.train_test_arrows(core_summary, CORE)
plt.show()

**Reading.** Every run earned more in the test window: from train to test, annualized return rises for all 8 runs, while annualized volatility rises for only 2 of the 8 runs (MaxSharpe and BL(0.1)).

## §7 H4 — volatility forecast bias

A *segment* is the holding period from one rebalance to the next. For each segment of the eight core runs, realized volatility is compared with the ex-ante (forecast) volatility the weights were chosen with, using notebook 07's segment construction.

In [ ]:
q_table, n_dropped, _ = h7.forecast_bias_table(panel, rdates, core, simple_returns, CORE)
n_seg = q_table.groupby("method").size().reindex(CORE)
assert n_seg.nunique() == 1, n_seg.to_dict()
q_mean = q_table.groupby("method")["q"].mean()
print(f"{int(n_seg.iloc[0])} segments per method ({n_dropped // len(CORE)} dropped per method)")

Mean variance ratio per method with its segment block-bootstrap standard error and interval, on the full and test windows.

In [ ]:
B4, BLOCK4, SEED4 = 10000, 12, 20260923
h4_rows, dec_rows = [], []
for m in CORE:
    seg = q_table[q_table["method"] == m].sort_values("asof")
    test = seg[seg["asof"] > split_ts]
    d = ml.robust.forecast_bias_decomposition(seg["realized_vol"], seg["exante_vol"])
    dt = ml.robust.forecast_bias_decomposition(test["realized_vol"], test["exante_vol"])
    assert abs(d["q_bar"] - float(q_mean[m])) <= 1e-12
    v_full = ((seg["realized_vol"] / seg["exante_vol"]) ** 2).to_numpy()
    draws = ml.robust.segment_block_bootstrap_mean(v_full, block=BLOCK4, B=B4, seed=SEED4)
    sd = float(draws.std(ddof=1))
    lo, hi = np.percentile(draws, [2.5, 97.5])
    h4_rows.append({"method": m, "vbar_full": d["vbar"], "sd": sd, "t": (d["vbar"] - 1) / sd,
                    "lo": float(lo), "hi": float(hi), "vbar_test": dt["vbar"],
                    "n_full": len(seg), "n_test": len(test)})
    for w, dd in (("full", d), ("test", dt)):
        dec_rows.append({"method": m, "window": w, "q_bar": dd["q_bar"],
                         "half_log_vbar": dd["half_log_vbar"], "J": dd["J"]})

h4 = pd.DataFrame(h4_rows).set_index("method")
h4_shown = h4.rename(columns={"vbar_full": "v̄ full", "sd": "bootstrap sd", "lo": "95% lo", "hi": "95% hi",
                              "vbar_test": "v̄ test", "n_full": "segments", "n_test": "segments in test"}).rename_axis(None)
h4_disp = ml.reporting.show_table(h4_shown, dp=2, int_cols=["segments", "segments in test"],
                                  col_dp={"v̄ full": 3, "v̄ test": 3})
h4_disp

Full-window mean variance ratio with its 95% segment block-bootstrap interval; the dashed line marks 1.

In [ ]:
f8a = h4.reset_index()[["method", "vbar_full", "lo", "hi"]]
fig = ml.plotting.forest_plot(f8a, "method", "vbar_full", "lo", "hi", ref=1.0,
                              xlabel="mean realized/ex-ante variance ratio (full window)")
plt.show()

**Reading.** The forecast is too low on average, not too high. v̄ is above 1 for all 8 methods, from 1.30 (EW) to 1.73 (MaxSharpe): on average realized variance has been larger than forecast, the opposite of the registered claim, and only MaxSharpe reaches an absolute t of 2 (+2.34), with the opposite sign. The 95% interval lies above 1 for GMV, MaxSharpe and MDP and contains 1 for the other 5; for GMV and MDP it excludes 1 while t < 2, because the bootstrap distribution of v̄ has a shorter lower tail than a normal one. The test-window v̄ is below 1 only for EW (0.90).

H4 descriptive: mean log ratio, half log mean variance ratio and Jensen gap per method, full and test windows.

In [ ]:
h4_dec = pd.DataFrame(dec_rows).set_index(["method", "window"])

H4_KEYS = {"EW": "EW", "GMV(S)": "GMV", "MaxSharpe(S)": "MaxSharpe", "BMV(0.3)": "BMV",
           "BL(0.1,S)": "BL", "MDP(S)": "MDP", "ERC(S)": "ERC", "HRP(S)": "HRP"}
assert list(H4_KEYS) == CORE
h4_stats = {}
for m, key in H4_KEYS.items():
    v, t, vt = float(h4.loc[m, "vbar_full"]), float(h4.loc[m, "t"]), float(h4.loc[m, "vbar_test"])
    h4_stats[f"H4_{key}"] = dict(display=f"vbar {v:.3f}, t {t:.2f}, test vbar {vt:.3f}",
                                 t_full=t, mean_test=vt - 1)
assert set(h4_stats) <= reg_ids
assert set(h4_stats) == {x["id"] for x in reg["hypothesis"] if x["id"].startswith("H4_") and x["kind"] == "directional_t"}

ml.reporting.show_table(h4_dec.rename(columns={"q_bar": "mean log ratio q̄", "half_log_vbar": "½·log v̄",
                                               "J": "Jensen gap J"}), dp=3)

Full-window decomposition of the mean log ratio into ½·log v̄ and the Jensen gap.

In [ ]:
fig = h.bias_decomposition_bars(h4_dec.xs("full", level="window").loc[CORE])
plt.show()

**Reading.** Notebook 07's negative mean log ratios come from the Jensen gap, not from an over-forecast. ½·log v̄ is positive for all 8 methods and q̄ is negative for 7 of the 8 methods (not MaxSharpe), so every negative q̄ comes from J: J runs from 0.20 to 0.37, ½·log v̄ from 0.13 to 0.27. For notebook 07's H7, the HRP − GMV difference in q̄ of −0.064 splits into +0.010 from ½·log v̄ and −0.074 from J; in variance terms the two are close (v̄ 1.53 for HRP, 1.50 for GMV; not tested).

### §7b Variance-ratio profile (descriptive, added after the H4 results)

The typical segment and the largest segments of v = (realized / ex-ante)² per core method, and v̄ without the segment starting 2020-02-29.

In [ ]:
VR_COLS = {"median_v": "median v", "share_v_lt_1": "share v < 1 (%)", "largest_asof": "largest segment start",
           "largest_v": "largest v", "second_asof": "second start", "second_v": "second v",
           "top5_share_sum_v": "top-5 share (%)", "vbar_ex_spike": "v̄ without the 2020 segment"}
VR_PCT = ["share v < 1 (%)", "top-5 share (%)"]
vr = h.variance_ratio_profile(q_table, CORE)
ml.reporting.show_table(vr.rename(columns=VR_COLS).rename_axis(None), pct=VR_PCT, int_cols=VR_PCT)

**Reading.** In most months the forecast is too high, and in a few stress months it is far too low. Median v is below 1 for all 8 methods (0.62 to 0.97) and 53–70% of segments have v < 1. The segment starting 2020-02-29 (March 2020) is the largest for every run (v from 37 for MDP to 74 for HRP), and the one starting 2025-03-31 (April 2025) is second for 7 of the 8 (MaxSharpe: 2018-01-31). The five largest segments carry 23–38% of the sum of v; without the 2020 segment alone v̄ falls by 0.17 to 0.35, to 0.98–1.42 (below 1 for EW).

**60/40 (exploratory, unregistered).** The same segments and profile for 60/40 (SPY/IEF, as in notebooks 01–08). It enters no test and no verdict.

In [ ]:
bench = h8.build_benchmarks(simple_returns, panel)
BENCH = h8.BENCH
for name, r in bench.items():
    assert r["wlog"].index.equals(rdates), f"{name}: wlog index != rdates"

q_bench, n_dropped_b, _ = h7.forecast_bias_table(panel, rdates, bench, simple_returns, BENCH)
assert n_dropped_b * len(CORE) == n_dropped, (n_dropped_b, n_dropped)
assert len(q_bench) == int(n_seg.iloc[0])
assert q_bench["asof"].tolist() == q_table.loc[q_table["method"] == CORE[0], "asof"].tolist()
d_b = ml.robust.forecast_bias_decomposition(q_bench["realized_vol"], q_bench["exante_vol"])
vr_b = h.variance_ratio_profile(q_bench, BENCH)
vr_b.insert(0, "vbar", d_b["vbar"])
vr_b["q_bar"] = d_b["q_bar"]
vr_b["J"] = d_b["J"]
vr_b_shown = vr_b.rename(columns={**VR_COLS, "vbar": "v̄", "q_bar": "mean log ratio q̄", "J": "Jensen gap J"}).rename_axis(None)
vr_b_disp = ml.reporting.show_table(vr_b_shown, pct=VR_PCT, int_cols=VR_PCT,
                                    col_dp={"mean log ratio q̄": 3, "Jensen gap J": 3})
vr_b_disp

**Reading.** 60/40 shows the same pattern as the core runs: the forecast is too high in the typical segment and too low on average. Its v̄ is 1.27, its median v is 0.60 and 71% of its segments have v < 1; its largest segment starts 2020-02-29 (v 43).

## Verdict table

The §1 rules applied to §5 (H3, 8 tests: Holm p < 0.05 and the same sign in the test window) and §7 (H4, 8 tests: t ≤ −2 and test-window v̄ below 1).

In [ ]:
table = ml.inference.evaluate(reg, {**h3_stats, **h4_stats})
assert len(table) == 16
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(table.set_index("item")[["statistic", "verdict"]])
is_h3 = table["item"].str.startswith("H3")
print(f"H3: {int((table.loc[is_h3, 'verdict'] == 'NULL HOLDS').sum())} of 8 NULL HOLDS; "
      f"H4: {int((table.loc[~is_h3, 'verdict'] == 'NOT SUPPORTED').sum())} of 8 NOT SUPPORTED")

## §8 TAKEAWAY — lookback changes turnover, not a tested Sharpe ratio; the volatility forecast is too low on average

- **H3: 8 of 8 nulls hold.** Every longer lookback has the higher full-window Sharpe (ΔSR +0.06 to +0.10), and all 8 keep their sign in the test window; the smallest Holm p is 0.338 (ERC 504d, raw t 2.03). Starting two years later moves the 252-day Sharpe ratios by −0.03 to −0.09, the same order as the lookback effect.
- **Lookback changes turnover and the 2022-12-31 UUP weight (descriptive).** From 252 to 756 days turnover falls to 41–57% of its 252-day level; the 2022-12-31 UUP weight falls for all 4 methods, each median UUP weight stays within 2.3 pp of its 252-day value, and UUP keeps the largest test-window median weight in all 12 grid runs.
- **H5 (descriptive).** Train and test Sharpe ranks correlate at ρ = 0.50; the largest rise is MaxSharpe from 6th to 1st, the largest fall BL(0.1) from 5th to 8th.
- **H4: 8 of 8 NOT SUPPORTED, and the evidence runs the other way.** v̄ runs from 1.30 (EW) to 1.73 (MaxSharpe); only MaxSharpe (t +2.34) reaches an absolute t of 2, with the sign opposite to the claim. ½·log v̄ is positive for every method, so each negative q̄ comes from the Jensen gap J (0.20 to 0.37). Notebook 07's H7 difference in q̄ of −0.064 splits into +0.010 from ½·log v̄ and −0.074 from J.
- **The forecast errs both ways (§7b, descriptive).** Median v is below 1 for all 8 methods, the segment starting 2020-02-29 is the largest for all 8 methods, and without it v̄ is 0.98 to 1.42 (below 1 for EW). 60/40, unregistered, shows the same pattern (v̄ 1.27, median v 0.60).

Lookback length changes turnover and the UUP weight at a date but no tested Sharpe ratio, and the ranking of the runs is only half stable from train to test. The 252-day volatility forecast is too high in the typical month and far too low in stress: the error the volatility overlay of notebook 10 reacts to.